# M1 · Detección de placas bolivianas + OCR
**IAM-811 Machine Learning y Aprendizaje Profundo · Laboratorio Final — Sistema Inteligente de Control de Peaje**

| Etapa | Modelo | Fuente |
|---|---|---|
| Detección de la placa | YOLOv9-t 384 end2end (ONNX) | `open-image-models` |
| Lectura (OCR) principal | `fast-plate-ocr` CCT-XS v2 global (ONNX) | entorno **venv** |
| Lectura comparativa 1 | Tesseract 5 (`--psm 11`, binarización Otsu) | entorno **conda** |
| Lectura comparativa 2 | EasyOCR (CRNN PyTorch) | entorno **mamba** |
| Validación | Regex boliviana `^\d{3,4}[A-Z]{3}$` + corrección por posición | — |

**Qué produce este notebook**
1. Descarga los modelos con **verificación SHA-256** (hashes oficiales de la guía `placas_entornos_v2.html`).
2. Evalúa los 3 lectores sobre las **fotos de placas bolivianas** del curso (las mismas que precarga el seeder de la BD).
3. Tabla de métricas: exactitud exacta (cruda y normalizada), CER, % formato válido y latencia.
4. Análisis de confusiones de caracteres (`3↔8`, `O↔0`, `I↔1`, `B↔8`, `S↔5`) que luego usa el agente conversacional.
5. *(Opcional)* Fine-tuning de un detector YOLO de placas bolivianas desde Roboflow.

> Entorno: Google Colab → *Entorno de ejecución → Cambiar tipo → GPU T4* (el OCR funciona también en CPU).

## 1. Instalación (versiones fijadas)

In [ ]:
!nvidia-smi -L || echo "Sin GPU (no es obligatoria para M1)"
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install "fast-plate-ocr[onnx]==1.1.0" onnxruntime==1.32.0 easyocr==1.7.2 pytesseract==0.3.13 \
               opencv-python-headless==5.1.0.95 rapidfuzz roboflow

## 2. Descarga de modelos con verificación SHA-256

In [ ]:
import hashlib, os, urllib.request
from pathlib import Path

MODELOS = Path('/content/modelos'); MODELOS.mkdir(exist_ok=True)
ARCHIVOS = {
    'yolo-v9-t-384-license-plates-end2end.onnx': (
        'https://github.com/ankandrew/open-image-models/releases/download/assets/yolo-v9-t-384-license-plates-end2end.onnx',
        '888397b96d761c89db40bc9c305838e8652660f5e282c2cadebbe8d2951a77a8'),
    'cct_xs_v2_global.onnx': (
        'https://github.com/ankandrew/cnn-ocr-lp/releases/download/arg-plates/cct_xs_v2_global.onnx',
        '8031afb5fdc6b4d80462c9d542f1284ebd2cfddf5dbacd62609848d7e2855f44'),
    'cct_xs_v2_global_plate_config.yaml': (
        'https://github.com/ankandrew/cnn-ocr-lp/releases/download/arg-plates/cct_xs_v2_global_plate_config.yaml',
        '0335c74a305173bb6f393efed0fde03cadeaa0b649ed8e19f431016d8232d0a6'),
}

def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

for nombre, (url, esperado) in ARCHIVOS.items():
    ruta = MODELOS / nombre
    if not ruta.exists():
        urllib.request.urlretrieve(url, ruta)
    real = sha256(ruta)
    assert real == esperado, f'SHA-256 NO coincide para {nombre}'
    print(f'OK  {nombre:42s} {real}')

## 3. Fotos de placas bolivianas (verdad de terreno)
Las fotos del curso están versionadas en el repositorio en `web/database/seeders/evidencias/` junto con
`manifest.csv` (`archivo, placa_real, categoria`). Se clona el repo público; también puede subir un `.zip` propio
con el mismo `manifest.csv`.

In [ ]:
import pandas as pd
REPO_URL = 'https://github.com/TU_USUARIO/sistema-peaje-ia.git'   # <- cambie por su repositorio público
DIR_FOTOS = Path('/content/sistema-peaje-ia/web/database/seeders/evidencias')

if not DIR_FOTOS.exists():
    !git clone -q --depth 1 {REPO_URL} /content/sistema-peaje-ia || echo "No se pudo clonar: suba un zip"
if not DIR_FOTOS.exists():
    from google.colab import files
    for n in files.upload():
        if n.endswith('.zip'):
            !unzip -qo "{n}" -d /content/fotos
    DIR_FOTOS = Path('/content/fotos')

manifest = pd.read_csv(DIR_FOTOS / 'manifest.csv', dtype=str).fillna('')
print(len(manifest), 'imágenes'); manifest.head(25)

## 4. Detector común + tres lectores

In [ ]:
import re, time, cv2, numpy as np, onnxruntime as ort
LADO = 384
sesion = ort.InferenceSession(str(MODELOS / 'yolo-v9-t-384-license-plates-end2end.onnx'), providers=['CPUExecutionProvider'])

def detectar(imagen, umbral=0.4):
    # Igual que detector.py de la guía: letterbox 384 + caja en coordenadas originales
    alto, ancho = imagen.shape[:2]
    escala = min(LADO / alto, LADO / ancho)
    nw, nh = round(ancho * escala), round(alto * escala)
    mx, my = (LADO - nw) / 2, (LADO - nh) / 2
    lienzo = cv2.copyMakeBorder(cv2.resize(imagen, (nw, nh)), round(my - 0.1), round(my + 0.1),
                                round(mx - 0.1), round(mx + 0.1), cv2.BORDER_CONSTANT, value=(114, 114, 114))
    x = (lienzo[:, :, ::-1].transpose(2, 0, 1) / 255.0).astype(np.float32)[None]
    cajas = []
    for _, x1, y1, x2, y2, _, c in sesion.run(None, {sesion.get_inputs()[0].name: x})[0]:
        if c < umbral: continue
        x1, x2 = [(v - mx) / escala for v in (x1, x2)]; y1, y2 = [(v - my) / escala for v in (y1, y2)]
        cajas.append((int(max(0, x1)), int(max(0, y1)), int(min(ancho, x2)), int(min(alto, y2)), float(c)))
    return sorted(cajas, key=lambda b: -(b[2] - b[0]) * (b[3] - b[1]) * b[4])

# --- Lector 1: fast-plate-ocr (venv) ---
from fast_plate_ocr import LicensePlateRecognizer
fpo = LicensePlateRecognizer(onnx_model_path=str(MODELOS / 'cct_xs_v2_global.onnx'),
                             plate_config_path=str(MODELOS / 'cct_xs_v2_global_plate_config.yaml'))
def leer_fastocr(rec):
    p = fpo.run_one(cv2.cvtColor(rec, cv2.COLOR_BGR2RGB), return_confidence=True)
    return p.plate.replace('_', '')

# --- Lector 2: Tesseract (conda) ---
import pytesseract
CFG = '--psm 11 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
def leer_tesseract(rec):
    b = cv2.resize(rec.max(axis=2), None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    b = cv2.bilateralFilter(b, 9, 75, 75)
    _, b = cv2.threshold(b, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    d = pytesseract.image_to_data(b, lang='eng', config=CFG, output_type=pytesseract.Output.DICT)
    pal = [(d['text'][i].strip(), d['height'][i]) for i in range(len(d['text'])) if d['text'][i].strip()]
    if not pal: return ''
    hmax = max(h for _, h in pal)
    return ''.join(t for t, h in pal if h >= 0.6 * hmax)

# --- Lector 3: EasyOCR (mamba) ---
import easyocr, torch
reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available(), verbose=False)
def leer_easyocr(rec):
    l = reader.readtext(rec, allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789')
    if not l: return ''
    alt = lambda r: max(p[1] for p in r[0]) - min(p[1] for p in r[0])
    m = max(alt(r) for r in l)
    l = sorted([r for r in l if alt(r) >= 0.6 * m], key=lambda r: r[0][0][0])
    return ''.join(t for _, t, _ in l)

LECTORES = {'fast-plate-ocr': leer_fastocr, 'tesseract': leer_tesseract, 'easyocr': leer_easyocr}

## 5. Normalización boliviana (idéntica a `inferencia/peaje_ia/normalizacion.py`)
El formato fija qué posiciones son dígitos (3 o 4 primeras) y cuáles letras (3 últimas), así que cada
carácter confundible se corrige según la posición: `O→0, I→1, B→8, S→5, Z→2, G→6` en dígitos y viceversa en letras.

In [ ]:
FORMATO = re.compile(r'^\d{3,4}[A-Z]{3}$')
A_DIG = {'O':'0','Q':'0','D':'0','U':'0','I':'1','L':'1','T':'1','J':'1','Z':'2','S':'5','B':'8','G':'6','A':'4','E':'3'}
A_LET = {'0':'O','1':'I','8':'B','5':'S','2':'Z','6':'G','4':'A','7':'T','3':'E'}
limpiar = lambda t: re.sub(r'[^A-Z0-9]', '', (t or '').upper())

def corregir(c, nd):
    out, cambios = [], 0
    for i, ch in enumerate(c):
        if i < nd:
            if ch.isdigit(): out.append(ch)
            elif ch in A_DIG: out.append(A_DIG[ch]); cambios += 1
            else: return None
        else:
            if ch.isalpha(): out.append(ch)
            elif ch in A_LET: out.append(A_LET[ch]); cambios += 1
            else: return None
    return ''.join(out), cambios

def normalizar(t):
    t = limpiar(t)
    if not t: return '', False
    if FORMATO.match(t): return t, True
    cand = []
    for largo, nd in ((7, 4), (6, 3)):
        for i in range(max(1, len(t) - largo + 1)):
            r = corregir(t[i:i + largo], nd) if len(t[i:i + largo]) == largo else None
            if r and FORMATO.match(r[0]):
                cand.append((r[1] + 1.5 * max(0, len(t) - largo), -largo, r[0], r[1]))
    if cand:
        cand.sort(); return cand[0][2], cand[0][3] <= 2
    return t, False

for p in ['2481-KLM', '1701N4C', 'I7O3XFY', 'BOLIVIA1534PHK']:
    print(p, '->', normalizar(p))

## 6. Evaluación sobre las fotos bolivianas

In [ ]:
from rapidfuzz.distance import Levenshtein

filas = []
for _, fila in manifest.iterrows():
    img = cv2.imread(str(DIR_FOTOS / fila['archivo']))
    if img is None: continue
    t0 = time.perf_counter(); cajas = detectar(img); t_det = (time.perf_counter() - t0) * 1000
    real = fila['placa_real']
    for nombre, lector in LECTORES.items():
        t0 = time.perf_counter()
        crudo = lector(img[cajas[0][1]:cajas[0][3], cajas[0][0]:cajas[0][2]]) if cajas else ''
        ms = (time.perf_counter() - t0) * 1000 + t_det
        norm, valido = normalizar(crudo)
        filas.append({'archivo': fila['archivo'], 'placa_real': real, 'lector': nombre,
                      'detectada': bool(cajas), 'crudo': limpiar(crudo), 'normalizada': norm, 'formato_valido': valido,
                      'acierto_crudo': real != '' and limpiar(crudo) == real,
                      'acierto_norm': real != '' and norm == real,
                      'cer': Levenshtein.distance(norm, real) / max(1, len(real)) if real else None,
                      'ms': ms})
res = pd.DataFrame(filas)
res.pivot_table(index=['archivo', 'placa_real'], columns='lector', values='normalizada', aggfunc='first')

In [ ]:
con_verdad = res[res.placa_real != '']
metricas = con_verdad.groupby('lector').agg(
    imagenes=('archivo', 'count'),
    deteccion_pct=('detectada', lambda s: 100 * s.mean()),
    exactitud_cruda_pct=('acierto_crudo', lambda s: 100 * s.mean()),
    exactitud_normalizada_pct=('acierto_norm', lambda s: 100 * s.mean()),
    formato_valido_pct=('formato_valido', lambda s: 100 * s.mean()),
    CER_medio=('cer', 'mean'),
    latencia_ms=('ms', 'mean'),
).round(2).sort_values('exactitud_normalizada_pct', ascending=False)
metricas.to_csv('/content/m1_metricas_ocr.csv'); res.to_csv('/content/m1_detalle_ocr.csv', index=False)
metricas

### Fotos ilegibles (deben quedar como `placa_ilegible` en el sistema)

In [ ]:
res[res.placa_real == ''][['archivo', 'lector', 'detectada', 'crudo', 'normalizada', 'formato_valido']]

## 7. Matriz de confusiones de caracteres (insumo del agente conversacional)

In [ ]:
from collections import Counter
conf = Counter()
for _, r in con_verdad[con_verdad.lector == 'fast-plate-ocr'].iterrows():
    for op, i, j in Levenshtein.editops(r['crudo'], r['placa_real']):
        if op == 'replace':
            conf[(r['crudo'][i], r['placa_real'][j])] += 1
pd.DataFrame([{'leido': a, 'real': b, 'veces': n} for (a, b), n in conf.most_common()]) \
  if conf else 'Sin sustituciones de caracteres'

In [ ]:
import matplotlib.pyplot as plt
muestras = manifest.sample(min(6, len(manifest)), random_state=1)
fig, ax = plt.subplots(2, 3, figsize=(15, 10)); ax = ax.ravel()
for a, (_, f) in zip(ax, muestras.iterrows()):
    img = cv2.imread(str(DIR_FOTOS / f['archivo']))
    for (x1, y1, x2, y2, c) in detectar(img)[:1]:
        txt = normalizar(leer_fastocr(img[y1:y2, x1:x2]))[0]
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 200, 0), 4)
        cv2.putText(img, f'{txt} {c:.2f}', (x1, max(30, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 200, 0), 3)
    a.imshow(img[:, :, ::-1]); a.set_title(f"real: {f['placa_real'] or 'ilegible'}"); a.axis('off')
plt.tight_layout(); plt.show()

## 8. (Opcional) Fine-tuning del detector de placas bolivianas con Roboflow
1. Cree un proyecto **Object Detection** en Roboflow con una clase `placa` y suba fotos de vehículos bolivianos
   (las del curso + capturas de video). Etiquete y genere una versión (*Generate → Export → YOLOv8*).
2. Guarde la API key en **Colab → 🔑 Secretos** con el nombre `ROBOFLOW_API_KEY` (nunca en el código).
3. Ejecute la celda. El modelo resultante se exporta a ONNX para reemplazar el detector genérico.

In [ ]:
ENTRENAR_DETECTOR = False      # cambie a True para entrenar
RF_WORKSPACE, RF_PROYECTO, RF_VERSION = 'tu-workspace', 'placas-bolivia', 1

if ENTRENAR_DETECTOR:
    !pip -q install ultralytics==8.4.218
    from google.colab import userdata
    from roboflow import Roboflow
    from ultralytics import YOLO
    ds = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY')).workspace(RF_WORKSPACE) \
            .project(RF_PROYECTO).version(RF_VERSION).download('yolov8')
    m = YOLO('yolo11n.pt')
    m.train(data=f'{ds.location}/data.yaml', epochs=60, imgsz=640, batch=16, project='/content/runs/placas', name='bo')
    best = YOLO('/content/runs/placas/bo/weights/best.pt')
    print(best.val(data=f'{ds.location}/data.yaml').box.map50)
    best.export(format='onnx', imgsz=640)

## 9. Descargar métricas

In [ ]:
from google.colab import files
files.download('/content/m1_metricas_ocr.csv'); files.download('/content/m1_detalle_ocr.csv')

### Conclusiones para el informe / video
* `fast-plate-ocr` (red entrenada solo con placas) supera a los OCR generales; Bolivia no está en su set de
  entrenamiento, por eso la **normalización por posición** sube la exactitud.
* Los errores residuales (p. ej. `3↔8`, `A↔4`) se resuelven en el **agente conversacional** con búsqueda difusa
  (Levenshtein ≤ 2) y la pregunta *"¿Quisiste decir …?"*.